In [ ]:

import socket
import sys
import errno
import os
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

def scan_port(host_ip, port):
    """Scan one TCP port and return its status."""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(0.5)
            result = sock.connect_ex((host_ip, port))

        if result == 0:
            return port, "open"

        # Connection refused means the port is closed.
        if result == errno.ECONNREFUSED or result == 10061:
            return port, "closed"

        # Recognize timeout errors on common operating systems.
        timeout_errors = {
            errno.ETIMEDOUT,
            getattr(errno, "EWOULDBLOCK", -1),
            getattr(errno, "EAGAIN", -1),
            10035,  # Windows: operation would block
            10060   # Windows: connection timed out
        }

        if result in timeout_errors:
            return port, "timeout"

        return port, f"error (code {result})"

    except socket.timeout:
        return port, "timeout"
    except OSError as error:
        return port, f"error ({error})"


def main():
    # Ask the user for the host and port range.
    host = input("Enter the hostname or IP address: ").strip()

    try:
        start_port = int(input("Enter the starting port (1-65535): "))
        end_port = int(input("Enter the ending port (1-65535): "))

        if not (1 <= start_port <= end_port <= 65535):
            print("Invalid port range. Please use ports 1-65535.")
            return

        host_ip = socket.gethostbyname(host)

    except ValueError:
        print("Please enter valid integer port numbers.")
        return
    except socket.gaierror:
        print("Could not resolve the hostname.")
        return
    except OSError as error:
        print(f"Could not resolve the host: {error}")
        return

    t1 = datetime.now()
    results = []

    print(f"\nStart time: {t1}")
    print(f"Scanning {host} ({host_ip})...")
    print("Finding port statuses: in progress...")

    # Scan multiple ports concurrently using threads.
    with ThreadPoolExecutor(max_workers=50) as executor:
        futures = [
            executor.submit(scan_port, host_ip, port)
            for port in range(start_port, end_port + 1)
        ]

        for future in as_completed(futures):
            results.append(future.result())

    results.sort(key=lambda item: item[0])

    t2 = datetime.now()
    total_time = t2 - t1

    # Save all results to the text file.
    with open("part-scanner.txt", "w", encoding="utf-8") as file:
        file.write(f"Host: {host}\n")
        file.write(f"IP address: {host_ip}\n")
        file.write(f"Start time: {t1}\n")
        file.write(f"Port range: {start_port}-{end_port}\n\n")

        for port, status in results:
            line = f"Port {port}: {status}"
            print(line)
            file.write(line + "\n")

        # Summarize the scan.
        open_count = sum(1 for _, status in results if status == "open")
        closed_count = sum(1 for _, status in results if status == "closed")
        timeout_count = sum(1 for _, status in results if status == "timeout")
        error_count = sum(1 for _, status in results if status.startswith("error"))

        file.write("\n--- Scan Summary ---\n")
        file.write(f"Open ports: {open_count}\n")
        file.write(f"Closed ports: {closed_count}\n")
        file.write(f"Timed-out ports: {timeout_count}\n")
        file.write(f"Other errors: {error_count}\n")
        file.write(f"End time: {t2}\n")
        file.write(f"Total time: {total_time}\n")

    print("\n--- Scan Summary ---")
    print(f"Open ports: {open_count}")
    print(f"Closed ports: {closed_count}")
    print(f"Timed-out ports: {timeout_count}")
    print(f"Other errors: {error_count}")
    print(f"End time: {t2}")
    print(f"Total time: {total_time}")
    print("Results saved to part-scanner.txt")


if __name__ == "__main__":
    main()
